In [ ]:
# 1. Configuration and attached-data discovery
from pathlib import Path
from collections import Counter
import os, re, json, math, shutil, hashlib, platform, subprocess

DATASET_URL = "https://www.kaggle.com/datasets/camboversecenter/clock-for-3dgs"
EXPECTED_INPUT = Path("/kaggle/input/datasets/camboversecenter/clock-for-3dgs/clock")
SITE = "wat-phnom-clock"

# Confirm these before publishing the trained heritage asset.
CAPTURE_CREATOR = "UNCONFIRMED"
CAPTURE_DATE = "UNCONFIRMED"
PERMISSION_REFERENCE = "UNCONFIRMED"
CONTENT_LICENSE = "UNCONFIRMED"

ITERATIONS = 7_000          # Preview first; change to 30_000 after visual approval.
TARGET_FRAMES = 354         # Curated source set; keep all overlapping views.
LONG_SIDE = 1_600           # Bounds COLMAP time and GPU memory.
MATCHING = "sequential"     # Frames came from ordered video paths.
MIN_REGISTERED_RATIO = 0.65
MIN_REGISTERED_IMAGES = 80

NORMAL_SPLATS = 180_000
ULTRA_SPLATS = 600_000
MAX_SPLAT_AXIS = 0.05
MIN_ALPHA = 0.15
RADIAL_CROP = 1.5

WORK = Path("/kaggle/working")
FRAMES = WORK / "frames"
PROC = WORK / "proc"
OUT = WORK / "out"
EXPORT = WORK / "export"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}

def natural_key(path: Path):
    parts = re.split(r"(\d+)", path.as_posix().lower())
    return [int(part) if part.isdigit() else part for part in parts]

if EXPECTED_INPUT.exists():
    data_root = EXPECTED_INPUT
else:
    roots = [p for p in Path("/kaggle/input").rglob("*") if p.is_dir()]
    ranked = [(sum(1 for f in p.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS), p) for p in roots]
    ranked = [item for item in ranked if item[0] > 0]
    assert ranked, "Clock dataset is not attached. Add camboversecenter/clock-for-3dgs."
    data_root = max(ranked)[1]

source_images = sorted(
    (p for p in data_root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS),
    key=natural_key,
)
assert len(source_images) >= MIN_REGISTERED_IMAGES, f"Only {len(source_images)} images found."
print("Dataset root:", data_root)
print("Images:", len(source_images))
print("Formats:", dict(Counter(p.suffix.lower() for p in source_images)))
print("First / last:", source_images[0].name, source_images[-1].name)


In [ ]:
# 2. Audit the curated clock images
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

def image_audit(path: Path) -> dict:
    with Image.open(path) as original:
        rgb = ImageOps.exif_transpose(original).convert("RGB")
        width, height = rgb.size
        preview = rgb.copy()
        preview.thumbnail((512, 512), Image.Resampling.LANCZOS)
        gray = np.asarray(preview.convert("L"), dtype=np.float32)
        focus = float(np.var(np.diff(gray, axis=0)) + np.var(np.diff(gray, axis=1)))
        dark = float(np.mean(gray < 8))
        bright = float(np.mean(gray > 247))
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    return {
        "path": str(path), "name": path.name, "width": width, "height": height,
        "megapixels": width * height / 1_000_000, "focus_score": focus,
        "dark_clip": dark, "bright_clip": bright, "sha256": digest,
    }

audit = pd.DataFrame(image_audit(path) for path in source_images)
audit["exact_duplicate"] = audit.duplicated("sha256", keep="first")
audit_path = WORK / f"{SITE}-image-audit.csv"
audit.to_csv(audit_path, index=False)

print("Exact duplicates:", int(audit.exact_duplicate.sum()))
print("Image sizes:", audit.groupby(["width", "height"]).size().to_dict())
print("Focus percentiles:", audit.focus_score.quantile([0, .05, .5, .95, 1]).round(2).to_dict())
print("Strongly clipped:", int(((audit.dark_clip > .20) | (audit.bright_clip > .20)).sum()))

sample_indices = np.linspace(0, len(source_images) - 1, 24).round().astype(int)
fig, axes = plt.subplots(4, 6, figsize=(16, 11))
for axis, index in zip(axes.flat, sample_indices):
    with Image.open(source_images[index]) as image:
        axis.imshow(ImageOps.exif_transpose(image).convert("RGB"))
    axis.set_title(f"{index}: {source_images[index].name}", fontsize=8)
    axis.axis("off")
plt.suptitle("Wat Phnom flower clock — capture audit", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# 3. Prepare ordered 1600px frames for COLMAP
usable = [Path(row.path) for row in audit.itertuples() if not row.exact_duplicate]
assert len(usable) >= MIN_REGISTERED_IMAGES
selected_indices = np.linspace(0, len(usable) - 1, min(TARGET_FRAMES, len(usable))).round().astype(int)
selected = [usable[index] for index in selected_indices]

shutil.rmtree(FRAMES, ignore_errors=True)
FRAMES.mkdir(parents=True)
for index, source in enumerate(selected):
    with Image.open(source) as original:
        image = ImageOps.exif_transpose(original).convert("RGB")
        image.thumbnail((LONG_SIDE, LONG_SIDE), Image.Resampling.LANCZOS)
        image.save(FRAMES / f"frame_{index:05d}.jpg", quality=95, optimize=True)

print(f"Prepared {len(selected)} / {len(usable)} images at <= {LONG_SIDE}px")


In [ ]:
# 4. Verify the Kaggle GPU before installing or training
import torch

assert torch.cuda.is_available(), "Enable a GPU in Notebook settings before continuing."
gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
print("Python:", platform.python_version())
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", gpu_name, "| capability:", gpu_capability)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)


In [ ]:
# 5. Install an isolated, compatible clock-training environment
from importlib.metadata import version as kernel_package_version
import sys, time

ENV_DIR = WORK / "clock-training-env"
NS_PY = ENV_DIR / "bin/python"
os.environ.update({
    "PYTHONUNBUFFERED": "1", "MAX_JOBS": "1", "OMP_NUM_THREADS": "4",
    "TORCH_CUDA_ARCH_LIST": f"{gpu_capability[0]}.{gpu_capability[1]}",
    "QT_QPA_PLATFORM": "offscreen", "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",
    "WANDB_MODE": "disabled",
})

def run_stage(stage, command):
    """Persist progress and errors even if the notebook output is truncated."""
    command = [str(part) for part in command]
    log_path = WORK / f"{SITE}-{stage}.log"
    status_path = WORK / f"{SITE}-run-status.json"
    status = {"stage": stage, "state": "running", "command": command,
              "started": time.time(), "log": str(log_path)}
    status_path.write_text(json.dumps(status, indent=2))
    print(f"\nSTAGE {stage}: {' '.join(command)}\nLog: {log_path}", flush=True)
    with log_path.open("a", buffering=1) as log:
        log.write("\nCOMMAND: " + " ".join(command) + "\n")
        process = subprocess.Popen(command, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            status.update(state="interrupted", finished=time.time())
            status_path.write_text(json.dumps(status, indent=2))
            raise
    status.update(state="complete" if returncode == 0 else "failed",
                  returncode=returncode, finished=time.time())
    status_path.write_text(json.dumps(status, indent=2))
    if returncode:
        raise RuntimeError(f"Stage {stage} failed ({returncode}). See {log_path}.")

run_stage("bootstrap", [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "uv", "plyfile"])
run_stage("colmap-packages", ["apt-get", "update", "-qq"])
run_stage("colmap-install", ["apt-get", "install", "-y", "-qq", "colmap"])
run_stage("python-runtime", [sys.executable, "-m", "uv", "python", "install", "3.10"])
if not NS_PY.exists():
    run_stage("create-environment", [sys.executable, "-m", "uv", "venv", "--python", "3.10", str(ENV_DIR)])
run_stage("pytorch-install", [sys.executable, "-m", "uv", "pip", "install", "--no-cache", "--python", str(NS_PY),
    "torch==2.7.1+cu128", "torchvision==0.22.1+cu128", "--index-url", "https://download.pytorch.org/whl/cu128"])
run_stage("nerfstudio-install", [sys.executable, "-m", "uv", "pip", "install", "--no-cache", "--python", str(NS_PY),
    "nerfstudio==1.1.5", "numpy==1.26.4", "gsplat==1.4.0", "plyfile", "setuptools<81"])
os.environ["PATH"] = str(ENV_DIR / "bin") + os.pathsep + os.environ["PATH"]
probe = """import json, torch, sys
from importlib.metadata import version
assert torch.cuda.is_available(), 'Isolated environment cannot access the GPU'
print(json.dumps({'python':sys.version.split()[0], 'torch':torch.__version__,
'cuda':torch.version.cuda, 'nerfstudio':version('nerfstudio'),
'gsplat':version('gsplat'), 'gpu':torch.cuda.get_device_name(0)}))
"""
TRAINING_SOFTWARE = json.loads(subprocess.check_output([str(NS_PY), "-c", probe], text=True).strip().splitlines()[-1])
def version(package):
    if package in TRAINING_SOFTWARE:
        return TRAINING_SOFTWARE[package]
    return kernel_package_version(package)
(WORK / f"{SITE}-training-environment.json").write_text(json.dumps(TRAINING_SOFTWARE, indent=2))
print("Training environment:", json.dumps(TRAINING_SOFTWARE, indent=2), flush=True)
run_stage("training-cli-check", [str(ENV_DIR / "bin/ns-train"), "splatfacto", "--help"])


In [ ]:
# 6. Estimate camera poses with COLMAP (slow stage)
process_command = [
    "ns-process-data", "images",
    "--data", str(FRAMES),
    "--output-dir", str(PROC),
    "--no-gpu",
    "--matching-method", MATCHING,
    "--num-downscales", "0",
]
print("Running:", " ".join(process_command))
run_stage("alignment", process_command)


In [ ]:
# 7. Registration gate — stop before training if the camera solve is weak
transforms = json.loads((PROC / "transforms.json").read_text())
registered = len(transforms["frames"])
registered_ratio = registered / len(selected)
print(f"COLMAP registered {registered} / {len(selected)} images ({registered_ratio:.1%}).")

assert registered >= MIN_REGISTERED_IMAGES, (
    f"Only {registered} cameras registered; inspect overlap and disconnected video paths."
)
assert registered_ratio >= MIN_REGISTERED_RATIO, (
    f"Registration ratio {registered_ratio:.1%} is below {MIN_REGISTERED_RATIO:.0%}."
)
print("Registration gate passed.")


In [ ]:
# 8. Train a 7k Splatfacto preview
# After visual approval, change ITERATIONS to 30_000 and rerun from a clean session.
train_command = [
    "ns-train", "splatfacto",
    "--data", str(PROC),
    "--output-dir", str(OUT),
    "--vis", "tensorboard",
    "--pipeline.datamanager.cache-images", "cpu",
    "--max-num-iterations", str(ITERATIONS),
]
print("Running:", " ".join(train_command))
run_stage("training", train_command)


In [ ]:
# 9. Evaluate held-out views and export the Gaussian PLY
configs = sorted(OUT.rglob("config.yml"), key=lambda path: path.stat().st_mtime)
assert configs, "Training completed without config.yml."
config_path = configs[-1]
metrics_path = WORK / f"{SITE}-metrics.json"

evaluation = subprocess.run([
    "ns-eval", "--load-config", str(config_path), "--output-path", str(metrics_path)
])
if evaluation.returncode:
    print("WARNING: ns-eval failed; require visual review before accepting the asset.")

shutil.rmtree(EXPORT, ignore_errors=True)
subprocess.run([
    "ns-export", "gaussian-splat",
    "--load-config", str(config_path),
    "--output-dir", str(EXPORT),
], check=True)
ply_candidates = list(EXPORT.glob("*.ply"))
assert ply_candidates, "ns-export produced no PLY."
ply_path = ply_candidates[0]
print("PLY:", ply_path, f"({ply_path.stat().st_size / 1e6:.1f} MB)")


In [ ]:
# 10. Convert PLY to the open antimatter15 .splat format
from plyfile import PlyData

SH_C0 = 0.28209479177387814

def ply_to_splat(ply_file: Path, output_file: Path, limit: int | None) -> dict:
    vertices = PlyData.read(ply_file)["vertex"].data
    xyz = np.stack([vertices["x"], vertices["y"], vertices["z"]], axis=1).astype(np.float32)
    scale = np.exp(np.stack([
        vertices["scale_0"], vertices["scale_1"], vertices["scale_2"]
    ], axis=1)).astype(np.float32)
    rotation = np.stack([
        vertices["rot_0"], vertices["rot_1"], vertices["rot_2"], vertices["rot_3"]
    ], axis=1).astype(np.float32)
    rotation /= np.linalg.norm(rotation, axis=1, keepdims=True) + 1e-12
    alpha = 1 / (1 + np.exp(-vertices["opacity"].astype(np.float32)))
    rgb = 0.5 + SH_C0 * np.stack([
        vertices["f_dc_0"], vertices["f_dc_1"], vertices["f_dc_2"]
    ], axis=1)

    finite = np.isfinite(xyz).all(axis=1) & np.isfinite(scale).all(axis=1)
    centroid = np.median(xyz[finite & (alpha >= MIN_ALPHA)], axis=0)
    valid = (
        finite & np.isfinite(rotation).all(axis=1)
        & (scale.max(axis=1) <= MAX_SPLAT_AXIS)
        & (alpha >= MIN_ALPHA)
        & (np.linalg.norm(xyz - centroid, axis=1) <= RADIAL_CROP)
    )
    importance = np.where(valid, alpha * np.cbrt(np.maximum(scale.prod(axis=1), 1e-18)), -1)
    order = np.argsort(-importance)[: int(valid.sum())]
    if limit is not None:
        order = order[:limit]

    assert len(order) > 0, "Filtering removed all Gaussians; preserve raw PLY and inspect thresholds."
    packed = np.zeros(len(order), dtype=[
        ("position", "<f4", 3), ("scale", "<f4", 3),
        ("color", "u1", 4), ("rotation", "u1", 4),
    ])
    packed["position"] = xyz[order]
    packed["scale"] = scale[order]
    rgba = np.concatenate([rgb[order], alpha[order, None]], axis=1)
    packed["color"] = (np.clip(rgba, 0, 1) * 255).round().astype(np.uint8)
    packed["rotation"] = np.clip(rotation[order] * 128 + 128, 0, 255).astype(np.uint8)
    packed.tofile(output_file)
    return {"file": output_file.name, "splats": len(order), "bytes": output_file.stat().st_size,
            "source_splats": len(vertices)}

normal_path = WORK / f"{SITE}.splat"
ultra_path = WORK / f"{SITE}-ultra.splat"
normal_stats = ply_to_splat(ply_path, normal_path, NORMAL_SPLATS)
ultra_stats = ply_to_splat(ply_path, ultra_path, ULTRA_SPLATS)
print("Normal:", normal_stats)
print("Ultra / VR:", ultra_stats)


In [ ]:
# 11. Package outputs with provenance and reproducibility metadata
import datetime as dt

final_ply = WORK / f"{SITE}.ply"
shutil.copy2(ply_path, final_ply)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

artifacts = [normal_path, ultra_path, final_ply, audit_path]
if metrics_path.exists():
    artifacts.append(metrics_path)

manifest = {
    "title": "Wat Phnom Flower Clock 3D Gaussian Splat",
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "source_dataset": DATASET_URL,
    "capture_creator": CAPTURE_CREATOR,
    "capture_date": CAPTURE_DATE,
    "permission_reference": PERMISSION_REFERENCE,
    "content_license": CONTENT_LICENSE,
    "pipeline": "Nerfstudio splatfacto via COLMAP sequential matching",
    "iterations": ITERATIONS,
    "input_images": len(source_images),
    "selected_images": len(selected),
    "registered_images": registered,
    "registered_ratio": registered_ratio,
    "normal": normal_stats,
    "ultra_vr": ultra_stats,
    "software": TRAINING_SOFTWARE,
    "files": {},
}
for artifact in artifacts:
    manifest["files"][artifact.name] = {"bytes": artifact.stat().st_size, "sha256": sha256(artifact)}

manifest_path = WORK / f"{SITE}-manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(manifest_path.read_text())

unconfirmed = [key for key, value in {
    "capture_creator": CAPTURE_CREATOR,
    "capture_date": CAPTURE_DATE,
    "permission_reference": PERMISSION_REFERENCE,
    "content_license": CONTENT_LICENSE,
}.items() if value == "UNCONFIRMED"]
if unconfirmed:
    print("PUBLICATION BLOCKED until confirmed:", ", ".join(unconfirmed))


In [ ]:
# 12. Final technical checks and downloadable bundle
assert normal_path.stat().st_size == normal_stats["splats"] * 32
assert ultra_path.stat().st_size == ultra_stats["splats"] * 32
assert normal_stats["splats"] <= NORMAL_SPLATS
assert ultra_stats["splats"] <= ULTRA_SPLATS
assert ultra_stats["splats"] >= normal_stats["splats"]

bundle_dir = WORK / f"{SITE}-deliverables"
shutil.rmtree(bundle_dir, ignore_errors=True)
bundle_dir.mkdir()
for artifact in [normal_path, ultra_path, final_ply, audit_path, manifest_path]:
    shutil.copy2(artifact, bundle_dir / artifact.name)
if metrics_path.exists():
    shutil.copy2(metrics_path, bundle_dir / metrics_path.name)
archive = shutil.make_archive(str(bundle_dir), "zip", root_dir=bundle_dir)

print(f"Normal: {normal_path.name} — {normal_path.stat().st_size / 1e6:.1f} MB")
print(f"Ultra/VR: {ultra_path.name} — {ultra_path.stat().st_size / 1e6:.1f} MB")
print("Bundle:", archive)
print("Technical checks passed. Visual inspection and low-end Android testing remain required.")
